# Mini-Projet : Analyse de Trafic Urbain (NYC Taxi Like)

Ce notebook est un projet complet qui simule une analyse Big Data de A à Z. Comptez environ 2 à 3 heures pour bien comprendre et explorer.

## Scénario
Vous êtes Data Engineer pour une startup de mobilité. On vous fournit des logs bruts de trajets de taxi. Votre mission :
1. Nettoyer les données (gestion des dates, des prix aberrants).
2. Enrichir les données (calculer la vitesse moyenne).
3. Produire des agrégats pour l'équipe BI (Revenu par heure, Zones les plus fréquentées).
4. Exporter les données propres.

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
import datetime

spark = SparkSession.builder.appName("NYC Taxi Mini Project").master("local[*]").getOrCreate()

## 1. Génération de Données Synthétiques (Massives)
Pour simuler le Big Data, nous allons générer un DataFrame de 10 000 lignes (imaginons 10 millions).

In [20]:
import random
import datetime
import builtins

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, TimestampType, DoubleType, StringType
)

# Si la session n'existe pas déjà
spark = SparkSession.builder.master("local[*]").appName("FakeData").getOrCreate()


def generate_fake_data(num_rows=10000):
    data = []
    zones = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]
    base_time = datetime.datetime(2023, 1, 1, 0, 0, 0)

    for i in range(num_rows):
        pickup = base_time + datetime.timedelta(minutes=random.randint(0, 10000))
        duration = random.randint(5, 120) # minutes
        dropoff = pickup + datetime.timedelta(minutes=duration)
        dist = builtins.round(random.uniform(1.0, 50.0), 2)
        fare = builtins.round(dist * 2.5 + 5 + random.uniform(0, 10), 2)
        if random.random() < 0.05: fare = -10.0 # Anomalie !
        zone = random.choice(zones)

        data.append((i, pickup, dropoff, dist, fare, zone))
    return data

schema = StructType([
    StructField("trip_id", IntegerType(), False),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropoff_datetime", TimestampType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("zone", StringType(), True)
])

raw_data = generate_fake_data()
df_raw = spark.createDataFrame(raw_data, schema)
df_raw.show(5)

+-------+-------------------+-------------------+-----------+-----------+--------+
|trip_id|    pickup_datetime|   dropoff_datetime|distance_km|fare_amount|    zone|
+-------+-------------------+-------------------+-----------+-----------+--------+
|      0|2023-01-07 01:13:00|2023-01-07 01:54:00|       9.63|      38.44|   Bronx|
|      1|2023-01-07 03:41:00|2023-01-07 04:40:00|       2.86|      15.35|Brooklyn|
|      2|2023-01-05 01:45:00|2023-01-05 03:29:00|       2.99|      21.14|Brooklyn|
|      3|2023-01-04 23:01:00|2023-01-04 23:26:00|      15.29|       44.6|   Bronx|
|      4|2023-01-03 06:50:00|2023-01-03 07:26:00|       7.79|      33.59|Brooklyn|
+-------+-------------------+-------------------+-----------+-----------+--------+
only showing top 5 rows


## 2. Nettoyage (Data Cleaning)
TACHE : Supprimez les courses avec un montant négatif et les distances nulles.

In [52]:
# A COMPLETER
df_raw.createOrReplaceTempView("trips")

df_clean = spark.sql("""
    SELECT *
    FROM trips
    WHERE fare_amount >= 0
      AND distance_km > 0
""")

df_clean.show(5)

+-------+-------------------+-------------------+-----------+-----------+--------+
|trip_id|    pickup_datetime|   dropoff_datetime|distance_km|fare_amount|    zone|
+-------+-------------------+-------------------+-----------+-----------+--------+
|      0|2023-01-07 01:13:00|2023-01-07 01:54:00|       9.63|      38.44|   Bronx|
|      1|2023-01-07 03:41:00|2023-01-07 04:40:00|       2.86|      15.35|Brooklyn|
|      2|2023-01-05 01:45:00|2023-01-05 03:29:00|       2.99|      21.14|Brooklyn|
|      3|2023-01-04 23:01:00|2023-01-04 23:26:00|      15.29|       44.6|   Bronx|
|      4|2023-01-03 06:50:00|2023-01-03 07:26:00|       7.79|      33.59|Brooklyn|
+-------+-------------------+-------------------+-----------+-----------+--------+
only showing top 5 rows


## 3. Feature Engineering
TACHE : Créez une colonne `duration_minutes` (différence entre dropoff et pickup) et une colonne `speed_kmh`.

In [55]:
# A COMPLETER
# Hint : Utilisez unix_timestamp pour la différence de temps
from pyspark.sql.functions import col, unix_timestamp

df_final = (
    df_clean
    .withColumn(
        "duration_minutes",
        (unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60
    )
    .withColumn(
        "speed_kmh",
        col("distance_km") / (col("duration_minutes") / 60)
    )
)

df_final.show(5)



+-------+-------------------+-------------------+-----------+-----------+--------+----------------+------------------+
|trip_id|    pickup_datetime|   dropoff_datetime|distance_km|fare_amount|    zone|duration_minutes|         speed_kmh|
+-------+-------------------+-------------------+-----------+-----------+--------+----------------+------------------+
|      0|2023-01-07 01:13:00|2023-01-07 01:54:00|       9.63|      38.44|   Bronx|            41.0| 14.09268292682927|
|      1|2023-01-07 03:41:00|2023-01-07 04:40:00|       2.86|      15.35|Brooklyn|            59.0|2.9084745762711863|
|      2|2023-01-05 01:45:00|2023-01-05 03:29:00|       2.99|      21.14|Brooklyn|           104.0|             1.725|
|      3|2023-01-04 23:01:00|2023-01-04 23:26:00|      15.29|       44.6|   Bronx|            25.0|            36.696|
|      4|2023-01-03 06:50:00|2023-01-03 07:26:00|       7.79|      33.59|Brooklyn|            36.0|12.983333333333334|
+-------+-------------------+-------------------

## 4. Analyse Business (Aggregations)
TACHE : Calculez le revenu moyen par Zone et par Heure de la journée (Extraire l'heure de pickup).

In [56]:
# A COMPLETER
df_analyse = df_final.groupBy("zone", hour("pickup_datetime").alias("pickup_hour")).mean("fare_amount"
)

df_analyse.show(5)

+---------+-----------+-----------------+
|     zone|pickup_hour| avg(fare_amount)|
+---------+-----------+-----------------+
| Brooklyn|         13|84.10761904761904|
|Manhattan|         22|73.10639534883721|
|   Queens|         12|76.44250000000001|
| Brooklyn|          7|78.23160000000001|
| Brooklyn|         18| 79.2075903614458|
+---------+-----------+-----------------+
only showing top 5 rows


## 5. Export
Sauvegardez le résultat final nettoyé en Parquet partitionné par 'Zone'.

In [57]:
df_final.write.partitionBy("zone").parquet("taxi_data_clean")